# Switch to a local model

So far our reviews went to a model in a data centre. **Open-weight models** can be downloaded and run on your own computer: no data leaves the machine, and there is no price per token. Ollama runs such models behind the same kind of API, so our code hardly changes.

In this notebook, we will:
1. Start Ollama and download a small model.
2. Send a first question to the local model, with and without thinking.
3. Classify 10 reviews with the local model and with `gpt-6-luna`.
4. Compare accuracy, speed and cost.

**Time:** about 15 minutes, self-study. Running all cells takes about 2 minutes, plus the download of the model (1.4 GB, once).

## Libraries and settings

In [ ]:
# Libraries
import os
import time
import platform
from datetime import datetime
from platform import python_version

import pandas as pd
from openai import OpenAI
from sklearn.metrics import accuracy_score

# Course helpers
from llm_course import DATA, ROOT, ask, parse_label

# API model and prices in USD per 1 million tokens
MODEL = 'gpt-6-luna'
PRICE_INPUT_PER_M = 0.10
PRICE_OUTPUT_PER_M = 0.50

# Local model: small enough for a codespace without a graphics card
LOCAL_MODEL = 'qwen3:1.7b'

# Check that the API key is available (never print the key itself)
print('API key found:', bool(os.environ.get('OPENAI_API_KEY')))

# Show current working directory
print(os.getcwd())

## Step 1: Start Ollama and download the model

The script installs Ollama if it is missing, starts it and downloads the model (about 1.4 GB, only the first time).

In [ ]:
!bash {ROOT}/scripts/start_ollama.sh {LOCAL_MODEL}

## Step 2: Talk to the local model

Ollama listens on this computer at `localhost:11434`. We use the same OpenAI library as before and change only the address (`base_url`). Ollama ignores the API key, but the library wants one.

In [ ]:
# A client that talks to Ollama on this computer
local = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')

def ask_local(prompt, effort='none'):
    """Send one prompt to the local model and return the answer text."""
    r = local.chat.completions.create(
        model=LOCAL_MODEL,
        messages=[{'role': 'user', 'content': prompt}],
        reasoning_effort=effort,
    )
    return r.choices[0].message.content

print(ask_local('Name one typical dish in the IKEA restaurant. One sentence.'))

### Thinking costs time

`qwen3` is a small reasoning model. With `effort='none'` it answers directly. With `effort='medium'` it first writes hidden thinking tokens, which takes much longer on a computer without a graphics card.

In [ ]:
question = 'Is this review positive, negative or mixed? Answer with one word: "Cheap furniture, rude staff."'

for effort in ['none', 'medium']:
    start = time.perf_counter()
    answer = ask_local(question, effort=effort)
    seconds = time.perf_counter() - start
    print(f'effort={effort:7s} {seconds:5.1f} seconds | answer: {answer}')

## Step 3: Classify 10 reviews with both models

We use 10 reviews with silver labels from notebook 03. Both models get the same prompt. Ten reviews keep the waiting time short: without a graphics card, the local model needs several seconds for a long review.

In [ ]:
# Reviews with silver labels from notebook 03
silver_path = DATA / 'processed/silver_labels.csv'
assert silver_path.exists(), 'Run notebook 03 first: it creates the silver labels.'
silver = pd.read_csv(silver_path)

# 10 random reviews
test = silver.sample(10, random_state=42).reset_index(drop=True)

PROMPT = ('Classify the sentiment of this customer review. '
          'Answer with exactly one word: positive, negative or mixed.\n\nReview: ')

test[['id', 'stars', 'silver', 'text']].head(5)

### Local model

`parse_label()` accepts an answer only if it is exactly one of our three labels. Everything else becomes `'unparsed'` and counts as an error.

In [ ]:
# Local model: one review after the other, and we stop the time
labels = []
start = time.perf_counter()

for text in test['text']:
    answer = ask_local(PROMPT + text)
    labels.append(parse_label(answer))

local_seconds = (time.perf_counter() - start) / len(test)
test['local'] = labels

print('Seconds per review:', round(local_seconds, 2))
test['local'].value_counts()

### API model

`use_cache=False` sends every request again instead of using a saved answer. We need this to measure the real speed. **Every run of this cell costs a little money** (far less than one cent).

In [ ]:
# API model: the same reviews, and we add up the tokens
labels = []
input_tokens = 0
output_tokens = 0
start = time.perf_counter()

for text in test['text']:
    r = ask(model=MODEL, input=PROMPT + text, reasoning={'effort': 'low'}, use_cache=False)
    labels.append(parse_label(r.output_text))
    input_tokens += r.usage.input_tokens
    output_tokens += r.usage.output_tokens

api_seconds = (time.perf_counter() - start) / len(test)
test['api'] = labels

# Cost: tokens / 1 million x price per 1 million tokens
api_cost = (input_tokens * PRICE_INPUT_PER_M + output_tokens * PRICE_OUTPUT_PER_M) / 1_000_000

print('Seconds per review:', round(api_seconds, 2))
print('Input tokens:', input_tokens, '| output tokens:', output_tokens)
test['api'].value_counts()

## Step 4: Compare accuracy, speed and cost

In [ ]:
comparison = pd.DataFrame(
    {
        LOCAL_MODEL: [accuracy_score(test['silver'], test['local']), local_seconds, 0.0],
        MODEL: [accuracy_score(test['silver'], test['api']), api_seconds, api_cost],
    },
    index=['accuracy (silver labels)', 'seconds per review', 'API cost in USD (10 reviews)'],
)
comparison.round(4)

In [ ]:
# Where do the two models disagree with the silver labels?
pd.set_option('display.max_colwidth', 300)
errors = test[(test['local'] != test['silver']) | (test['api'] != test['silver'])]
errors[['id', 'silver', 'local', 'api', 'text']]

## Interpret the comparison

- **Accuracy:** the local model has 1.7 billion parameters, far fewer than the API model. It makes more mistakes, especially with `mixed`.
- **Fair comparison?** The silver labels were made by `gpt-6-luna` itself. A model agrees with its own labels more easily; a fair comparison needs gold labels made by people.
- **10 reviews are few:** one review more or less changes the accuracy by 10 percentage points. For a real decision you would test many more.
- **Cost:** the local model has no price per token, but it is not free: it needs hardware, electricity and somebody who runs it.
- **Privacy:** with a local model, the reviews never leave the computer. That can be the deciding argument.

**Your turn:**
1. Test more reviews: change `sample(10, ...)` to `sample(30, ...)` in step 3 and run the cells again. Does the picture change?
2. Let the local model think: replace `ask_local(PROMPT + text)` with `ask_local(PROMPT + text, effort='medium')` in the cell *Local model* and run the cells again. What happens to accuracy and to seconds per review? (This takes several minutes.)
3. One million reviews per month: how long would the local model need in this codespace, and what would the API cost? Use the numbers from the table.
4. Is a language model the right tool for this task at all? Which simpler method from earlier weeks could you try first?

### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [ ]:
# Show operating system, date/time and Python version
print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')